# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2k24csaiml1e2411265-wq/ML_starter_FlyrankAI/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Method:** Random Forest Classifier.

I chose Random Forest because the lane has a binary decline outcome and numeric search-performance signals. It can capture non-linear relationships and interactions while remaining practical to inspect with permutation importance. I compare it with the Week-4 baseline on the same held-out client groups and metric. Complexity is useful only if it improves the honest result.

In [1]:
%pip -q install duckdb huggingface_hub pandas numpy scikit-learn
import os,getpass,duckdb,pandas as pd,numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,classification_report
from sklearn.inspection import permutation_importance
HF_TOKEN=os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata; HF_TOKEN=userdata.get('HF_TOKEN')
    except Exception: pass
HF_TOKEN=HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
assert HF_TOKEN
con=duckdb.connect(); con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL='hf://datasets/FlyRank/internship-warehouse'
DAILY=f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
QUERY=f"read_parquet('{REL}/fact_content_query_90d.parquet')"
print('Connected.')

Paste your Hugging Face READ token (hf_...): ··········
Connected.


## 2. Split design

I use **GroupShuffleSplit on `client_hash_id`**. A client is entirely in either train or test, preventing rows from the same client appearing in both sets. This tests cross-client generalization and is more honest than a random row split for this question. The same held-out rows are used for the baseline and Random Forest.

In [2]:
features=con.sql(f"""
WITH base AS (
 SELECT client_hash_id,content_hash_id,SUM(gsc_impressions) imp_prev30,SUM(gsc_clicks) clk_prev30,
 AVG(gsc_avg_position) pos_prev30,STDDEV_SAMP(gsc_avg_position) pos_volatility
 FROM {DAILY} WHERE report_date>=DATE '2026-02-01' AND report_date<DATE '2026-03-01'
 GROUP BY 1,2 HAVING SUM(gsc_impressions)>=100),
queries AS (SELECT content_hash_id,SUM(impressions_90d) kept_impressions,MAX(impressions_90d) top_query_impressions FROM {QUERY} GROUP BY content_hash_id),
outcome AS (SELECT client_hash_id,content_hash_id,SUM(gsc_impressions) imp_future30 FROM {DAILY} WHERE report_date>=DATE '2026-03-01' AND report_date<DATE '2026-04-01' GROUP BY 1,2)
SELECT b.*,CASE WHEN b.imp_prev30>0 THEN b.clk_prev30/b.imp_prev30 ELSE NULL END ctr_prev30,
CASE WHEN q.kept_impressions>0 THEN q.top_query_impressions/q.kept_impressions ELSE NULL END query_concentration,o.imp_future30
FROM base b LEFT JOIN queries q USING(content_hash_id) INNER JOIN outcome o USING(client_hash_id,content_hash_id)
""").df()
features['is_declining']=(features['imp_future30']<0.8*features['imp_prev30']).astype(int)
feature_cols=['imp_prev30','clk_prev30','ctr_prev30','pos_prev30','pos_volatility','query_concentration']
model_data=features.dropna(subset=['client_hash_id','is_declining']).copy()
print('Rows:',len(model_data),'Clients:',model_data.client_hash_id.nunique(),'Decline rate:',round(model_data.is_declining.mean(),3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 76837 Clients: 34 Decline rate: 0.182


In [3]:
gss=GroupShuffleSplit(n_splits=1,test_size=0.25,random_state=42)
train_idx,test_idx=next(gss.split(model_data,model_data['is_declining'],groups=model_data['client_hash_id']))
train=model_data.iloc[train_idx].copy(); test=model_data.iloc[test_idx].copy()
print('Train rows:',len(train),'Test rows:',len(test))
print('Train clients:',train.client_hash_id.nunique(),'Test clients:',test.client_hash_id.nunique())
print('Client overlap:',len(set(train.client_hash_id)&set(test.client_hash_id)))
assert len(set(train.client_hash_id)&set(test.client_hash_id))==0

Train rows: 68338 Test rows: 8499
Train clients: 25 Test clients: 9
Client overlap: 0


## 3. Train + compare vs my baseline

**Week-4 baseline:** a transparent low-volume review rule using only the previous-30-day impression signal. Both methods use exactly the same held-out client groups. The primary comparison metric is **F1**, with precision, recall, accuracy, and ROC-AUC shown where applicable.

In [4]:
BASE_THRESHOLD=500
y_test=test['is_declining'].to_numpy()
baseline_pred=(test['imp_prev30']<BASE_THRESHOLD).astype(int).to_numpy()
def row(name,y,p):
 return {'method':name,'F1':f1_score(y,p,zero_division=0),'precision':precision_score(y,p,zero_division=0),'recall':recall_score(y,p,zero_division=0),'accuracy':accuracy_score(y,p)}

In [5]:
X_train=train[feature_cols]; X_test=test[feature_cols]; y_train=train['is_declining']
rf=Pipeline([('imputer',SimpleImputer(strategy='median')),('model',RandomForestClassifier(n_estimators=250,max_depth=8,min_samples_leaf=5,class_weight='balanced',random_state=42,n_jobs=-1))])
rf.fit(X_train,y_train); model_pred=rf.predict(X_test); model_prob=rf.predict_proba(X_test)[:,1]
results=pd.DataFrame([row('Week-4 baseline',y_test,baseline_pred),row('Random Forest',y_test,model_pred)])
results['ROC-AUC']=[np.nan,roc_auc_score(y_test,model_prob)]
display(results); print(classification_report(y_test,model_pred,digits=3,zero_division=0))

,method,F1,precision,recall,accuracy,ROC-AUC
0,Week-4 baseline,0.262977,0.167396,0.612989,0.545594,NaN
1,Random Forest,0.299420,0.185256,0.780249,0.517120,0.676868


              precision    recall  f1-score   support

           0      0.934     0.477     0.632      7375
           1      0.185     0.780     0.299      1124

    accuracy                          0.517      8499
   macro avg      0.560     0.629     0.466      8499
weighted avg      0.835     0.517     0.588      8499



## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [6]:
test_view=test[['client_hash_id','content_hash_id']+feature_cols+['is_declining']].copy(); test_view['prediction']=model_pred
test_view['error_type']=np.select([(test_view.is_declining==1)&(test_view.prediction==0),(test_view.is_declining==0)&(test_view.prediction==1)],['false_negative','false_positive'],default='correct')
print('Error counts:'); display(test_view.error_type.value_counts().to_frame('n'))
print('Mean feature values by error type:'); display(test_view.groupby('error_type')[feature_cols].mean(numeric_only=True).round(3))

Error counts:


,n
error_type,
correct,4395
false_positive,3857
false_negative,247


Mean feature values by error type:


,imp_prev30,clk_prev30,ctr_prev30,pos_prev30,pos_volatility,query_concentration
error_type,,,,,,
correct,1659.397,6.569,0.004,11.441,5.048,0.386
false_negative,1244.206,7.591,0.010,10.049,4.439,0.428
false_positive,864.104,1.808,0.002,9.731,6.029,0.675


In [7]:
perm=permutation_importance(rf,X_test,y_test,scoring='f1',n_repeats=5,random_state=42,n_jobs=-1)
importance=pd.DataFrame({'feature':feature_cols,'mean_f1_drop':perm.importances_mean,'std':perm.importances_std}).sort_values('mean_f1_drop',ascending=False)
display(importance); print('Most influential signal:',importance.iloc[0]['feature'])

,feature,mean_f1_drop,std
5,query_concentration,0.076768,0.005183
3,pos_prev30,0.014918,0.002356
2,ctr_prev30,0.012434,0.001583
4,pos_volatility,0.005648,0.001537
1,clk_prev30,-0.001490,0.000996
0,imp_prev30,-0.003122,0.001256


Most influential signal: query_concentration


### Error interpretation

- **False positives:** the model requests review where the defined >20% decline did not occur; these reduce precision and consume review capacity.
- **False negatives:** a real measured decline was missed; these are important misses for a refresh workflow.
- **Feature importance:** permutation importance is directional, not causal. It shows which signals matter most to held-out F1 in this experiment.
- **Generalization:** client-grouped validation tests transfer across clients rather than memorization of client-specific rows.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.